<style>
.drift-hero { background: radial-gradient(circle at 18% 5%, #155e75, transparent 38%), linear-gradient(135deg,#07111f,#172554 58%,#3b0764); border-radius: 28px; color:#eef8ff; padding: 42px 46px; margin: 8px 0 28px; box-shadow: 0 22px 60px rgba(15,23,42,.28); }
.drift-kicker { color:#67e8f9; font: 700 12px/1.2 system-ui; letter-spacing:.18em; text-transform:uppercase; }
.drift-hero h1 { font: 800 52px/1.05 system-ui; letter-spacing:-.035em; margin:14px 0 14px; overflow-wrap:anywhere; }
.drift-hero p { color:#cbd5e1; font: 18px/1.6 system-ui; max-width:720px; margin:0; }
.drift-badges { margin-top:22px; display:flex; gap:8px; flex-wrap:wrap; }
.drift-badge { border:1px solid rgba(103,232,249,.45); border-radius:999px; color:#cffafe; font:600 12px system-ui; padding:6px 10px; }
.drift-grid { display:grid; grid-template-columns:repeat(4,minmax(0,1fr)); gap:12px; margin:18px 0 26px; }
.drift-card { border:1px solid #dbeafe; border-radius:16px; background:#f8fbff; padding:16px; }
.drift-card strong { display:block; color:#0f172a; font:700 14px system-ui; margin-bottom:6px; }
.drift-card span { color:#475569; font:13px/1.45 system-ui; }
.drift-panel { border:1px solid #bfdbfe; border-radius:18px; background:#f8fbff; padding:20px; margin:14px 0; }
.drift-panel h3 { color:#0f172a; font:700 17px system-ui; margin:0 0 10px; }
.drift-muted { color:#64748b; font:13px/1.5 system-ui; }
.drift-ok { color:#047857; font:700 13px system-ui; }
.drift-warn { color:#9a3412; font:700 13px system-ui; }
.drift-table { width:100%; border-collapse:collapse; font:13px system-ui; }
.drift-table th { color:#475569; font-size:11px; letter-spacing:.08em; text-align:left; text-transform:uppercase; padding:9px; border-bottom:1px solid #cbd5e1; }
.drift-table td { color:#1e293b; padding:10px 9px; border-bottom:1px solid #e2e8f0; }
.drift-claim { border-left:4px solid #38bdf8; border-radius:8px; background:#f8fafc; margin:10px 0; padding:12px 14px; }
.drift-claim .kind { color:#0369a1; font:700 11px system-ui; letter-spacing:.08em; text-transform:uppercase; }
.drift-claim .text { color:#0f172a; font:14px/1.5 system-ui; margin:4px 0 8px; }
.drift-claim .quote { color:#475569; font:italic 13px/1.45 system-ui; }
@media(max-width:800px){.drift-grid{grid-template-columns:1fr 1fr}.drift-hero{padding:30px}.drift-hero h1{font-size:40px}}
</style>

<section class="drift-hero">
  <div class="drift-kicker">DRIFT · evidence run console</div>
  <h1>DRIFT Manual Run</h1>
  <p>Turn a bounded set of primary GPU and infrastructure releases into inspectable engineering evidence—then publish nothing until a human has reviewed every claim.</p>
  <div class="drift-badges"><span class="drift-badge">Primary-source only</span><span class="drift-badge">Luna draft + verifier</span><span class="drift-badge">Human review gate</span><span class="drift-badge">SHA-256 evidence archive</span></div>
</section>

<div class="drift-grid">
<div class="drift-card"><strong>1 · Observe</strong><span>Read configured first-party release feeds. Source text remains data, never instructions.</span></div>
<div class="drift-card"><strong>2 · Ground</strong><span>Freeze exact excerpts, offsets, hashes, and upstream PR/commit references for every claim.</span></div>
<div class="drift-card"><strong>3 · Review</strong><span>Keep model output quarantined until a reviewer records meaningful notes.</span></div>
<div class="drift-card"><strong>4 · Prove</strong><span>Write a scrubbed, immutable evidence record and SHA-256 manifest for the reviewed run.</span></div>
</div>

### The judge-facing contract

DRIFT promises traceable primary facts, explicitly labelled interpretation, and bounded checks. It does **not** claim compatibility certification, incident prediction, or that a model verifier proves a release claim.

## 0. Preflight — prove the run is safe to start

This cell checks mode, budget, API-key presence, and the effective database host without revealing a key, password, or full connection string. A public Railway TCP proxy may replace only the host/port of the private service URL.

In [ ]:
import os
import sys
from html import escape
from pathlib import Path
from urllib.parse import urlsplit

from IPython.display import HTML, Markdown, display


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'backend' / 'sources.yaml').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook from the DRIFT repository or a child directory.')


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
os.chdir(REPOSITORY_ROOT)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from backend.core.config import settings

settings.validate()
database_host = urlsplit(settings.database_url).hostname or "unknown"
if settings.mode != "live":
    raise RuntimeError("Set DRIFT_MODE=live before a model-backed capture.")
if "postgres.railway.internal" in database_host:
    raise RuntimeError("The private Railway hostname is still active. Configure a public proxy override or local PostgreSQL.")

checks = [
    ("Live mode", settings.mode == "live", settings.mode),
    ("API key present", bool(settings.openai_api_key), "configured" if settings.openai_api_key else "missing"),
    ("Database route", bool(database_host), database_host),
    ("Project budget", True, f"${settings.max_spend_usd:.2f} cap · ${settings.max_call_usd:.2f} per-attempt reservation"),
]
rows = ''.join(f"<tr><td><span class='drift-ok'>● ready</span></td><td>{escape(label)}</td><td class='drift-muted'>{escape(detail)}</td></tr>" for label, ok, detail in checks if ok)
display(HTML(f"<section class='drift-panel'><h3>Run readiness</h3><table class='drift-table'><tbody>{rows}</tbody></table><p class='drift-muted'>No provider request has been made.</p></section>"))
print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Spend ledger: {Path(settings.spend_ledger_path)}")

## 1. Source roster — what this run will observe

Every source is a configured primary GitHub release feed. This notebook starts with one recent item from each enabled source and caps the total at eight.

In [ ]:
from backend.agents.scout import load_sources

sources = [source for source in load_sources() if source.get("enabled", True)]
source_rows = ''.join(
    f"<tr><td><strong>{escape(source['id'])}</strong></td><td>{escape(source['name'])}</td><td>{escape(source['category'])}</td><td><a href='{escape(source['feed_url'], quote=True)}' target='_blank'>primary feed ↗</a></td></tr>"
    for source in sources
)
display(HTML(f"<section class='drift-panel'><h3>{len(sources)} primary sources ready</h3><table class='drift-table'><thead><tr><th>ID</th><th>Project</th><th>Category</th><th>Evidence</th></tr></thead><tbody>{source_rows}</tbody></table></section>"))

## 2. Configure the bounded run

Luna drafts and separately verifies claims. Depending on clustering, this all-source run may make up to **24 structured model calls** (classification, draft, verifier for each source) and two embedding calls. The spend guard reserves a retry envelope and settles reported usage; it blocks the project ceiling.

Set `CONFIRM_CAPTURE = "RUN"` only after you accept the provider spend.

In [ ]:
from datetime import UTC, datetime

from backend.core.model_router import Tier

SOURCE_IDS = {source['id'] for source in sources}
PER_SOURCE_LIMIT = 1
MAX_ITEMS = 8
TIER = Tier.DEV
RUN_LABEL = f"{datetime.now(UTC).date().isoformat()}-all-sources-reviewed"
CONFIRM_CAPTURE = ""  # Set to "RUN" only after you accept the provider spend.

if not SOURCE_IDS or PER_SOURCE_LIMIT < 1 or MAX_ITEMS < 1:
    raise ValueError("Choose at least one source and positive limits.")
max_structured_calls = len(SOURCE_IDS) * 3
display(HTML(f"<section class='drift-panel'><h3>Bounded all-source plan</h3><p class='drift-muted'>Sources: <strong>{', '.join(sorted(SOURCE_IDS))}</strong><br>Selection: <strong>{PER_SOURCE_LIMIT} per source · {MAX_ITEMS} total</strong><br>Model tier: <strong>{TIER.value} / Luna</strong><br>Potential provider operations: <strong>up to {max_structured_calls} structured + 2 embedding calls</strong><br>Evidence archive name: <strong>{RUN_LABEL}</strong></p><p class='drift-warn'>Capture confirmation: {escape(CONFIRM_CAPTURE or 'not set')} — no provider call until this equals RUN.</p></section>"))

## 3. Capture — drafts only

This runs Scout → Synthesizer → claim extraction → separate verifier. Successful records remain private drafts; nothing appears in live briefing, search, or chat yet.

In [ ]:
from backend.pipeline import run_capture

if CONFIRM_CAPTURE != "RUN":
    display(HTML("<section class='drift-panel'><h3>Capture paused</h3><p class='drift-warn'>Set CONFIRM_CAPTURE = 'RUN' in the previous cell, re-run it, then run this cell.</p></section>"))
else:
    capture = await run_capture(
        source_ids=SOURCE_IDS,
        per_source_limit=PER_SOURCE_LIMIT,
        max_items=MAX_ITEMS,
        tier=TIER,
    )
    display(HTML(f"<section class='drift-panel'><h3>Draft capture complete</h3><p class='drift-ok'>Fetched {capture.fetched_count} · selected {capture.selected_count} · persisted raw evidence {capture.persisted_raw_count} · draft Insight IDs {capture.insight_ids}</p><p class='drift-muted'>These records are quarantined until the review step below.</p></section>"))

## 4. Inspect — make every claim earn trust

Read the exact source excerpts. Direct facts, conditional interpretations, and recommended checks must remain visibly distinct. The verifier is screening—not proof.

In [ ]:
from sqlalchemy import select

from backend.models.schema import InsightRow, session_factory

if 'capture' not in globals():
    display(HTML("<section class='drift-panel'><h3>Nothing to inspect yet</h3><p class='drift-muted'>Complete the draft capture first.</p></section>"))
else:
    async with session_factory() as session:
        draft_rows = (await session.scalars(select(InsightRow).where(InsightRow.id.in_(capture.insight_ids)))).all()
    for row in draft_rows:
        claims = ''.join(
            f"<article class='drift-claim'><div class='kind'>{escape(claim['kind'])}</div><div class='text'>{escape(claim['text'])}</div>" + ''.join(
                f"<div class='quote'>“{escape(evidence['excerpt'])}”<br><a href='{escape(evidence['source_url'], quote=True)}' target='_blank'>open primary source ↗</a> · chars {evidence['start_char']}–{evidence['end_char']}</div>"
                for evidence in claim['evidence']
            ) + "</article>"
            for claim in row.claims
        )
        risks = ', '.join(row.operator_risks) or 'none labelled'
        display(HTML(f"<section class='drift-panel'><h3>Draft {row.id} · {escape(row.title)}</h3><p class='drift-muted'>Verifier: <strong>{escape(row.verification_status)}</strong> · upstream release type: <strong>{escape(row.upstream_release_type)}</strong> · potential operator risks: <strong>{escape(risks)}</strong></p>{claims}</section>"))

## 5. Human review — the publication gate

Enter only draft IDs you personally checked against the source excerpts. Notes must state what was reviewed. This is the only step that makes a record eligible for live `/briefing`, `/search`, and `/chat`.

In [ ]:
from backend.review import publish_verified_insights

PUBLISH_IDS: list[int] = []  # e.g. [12, 13]
REVIEW_NOTES = ""  # Describe the source/claim/action checks you completed.

if PUBLISH_IDS:
    async with session_factory() as session:
        published = await publish_verified_insights(session, PUBLISH_IDS, review_notes=REVIEW_NOTES)
    display(HTML(f"<section class='drift-panel'><h3>Published after human review</h3><p class='drift-ok'>Reviewed Insight IDs: {published}</p></section>"))
else:
    display(HTML("<section class='drift-panel'><h3>Review gate closed</h3><p class='drift-warn'>No record is public. Add personally reviewed IDs and meaningful notes only when ready.</p></section>"))

## 6. Evidence archive — leave a reproducible proof

Archive only published IDs from this run. The output contains public Insight fields, typed claims, frozen excerpts, and capture counts—but never review notes, keys, connection strings, or spend-ledger contents. A SHA-256 manifest prevents silent replacement.

In [ ]:
from backend.core.live_store import _row_to_insight
from backend.evidence_archive import archive_reviewed_capture

ARCHIVE_IDS: list[int] = []  # reviewed IDs only, e.g. [12, 13]
ARCHIVE_NAME = RUN_LABEL

if ARCHIVE_IDS:
    if 'capture' not in globals():
        raise RuntimeError("Archive only evidence created by a completed notebook capture.")
    async with session_factory() as session:
        archive_rows = (await session.scalars(select(InsightRow).where(InsightRow.id.in_(ARCHIVE_IDS)))).all()
    if {row.id for row in archive_rows} != set(ARCHIVE_IDS):
        raise ValueError("Every ARCHIVE_ID must exist before evidence is written.")
    archive = archive_reviewed_capture(
        [_row_to_insight(row) for row in archive_rows],
        archive_name=ARCHIVE_NAME,
        capture_metadata={
            'source_ids': sorted(SOURCE_IDS),
            'fetched_count': capture.fetched_count,
            'selected_count': capture.selected_count,
            'persisted_raw_count': capture.persisted_raw_count,
            'draft_insight_ids': capture.insight_ids,
        },
    )
    display(HTML(f"<section class='drift-panel'><h3>Evidence archived</h3><p class='drift-ok'>JSON: {escape(str(archive.evidence_path))}<br>Manifest: {escape(str(archive.manifest_path))}<br>SHA-256: <code>{archive.sha256}</code></p></section>"))
else:
    display(HTML("<section class='drift-panel'><h3>Archive waiting for review</h3><p class='drift-warn'>No file written. Archive only reviewed, published IDs.</p></section>"))

---

### What a completed DRIFT Manual Run proves

- The selected primary releases were captured within explicit limits.
- Every public claim is traceable to a frozen primary-source excerpt.
- Model reasoning is labelled and separately screened.
- A human—not the model—approved publication.
- The reviewed result is saved as a dated, checksummed evidence artifact.

That is useful release intelligence without pretending to be a compatibility guarantee.